# 02 — Nettoyage et normalisation de la base SQLite

Objectif : transformer les données brutes en données exploitables pour le scoring et l’interface, sans perdre les signaux métier utiles.

Cette étape conserve les données sources intactes, documente les anomalies observées, applique des règles de nettoyage explicites, et prépare des tables normalisées pour la suite du projet.

In [1]:
# Import des bibliothèques nécessaires pour le nettoyage et la normalisation
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

# Chemin vers la base brute
db_path = Path("H:/Ecole/ECE/CESURE/ALTERNANCE/Sharesub.com/TEST_TECHNIQUE/risk-monitor/data/raw/risk_monitor_dataset.sqlite")

# Vérification préalable du fichier source
assert db_path.exists(), f"Fichier introuvable : {db_path}"

# Ouverture de la connexion SQLite brute
conn = sqlite3.connect(db_path)

# Chargement des tables dans des DataFrames pour travailler proprement en mémoire
users = pd.read_sql_query("SELECT * FROM users;", conn)
subscriptions = pd.read_sql_query("SELECT * FROM subscriptions;", conn)
memberships = pd.read_sql_query("SELECT * FROM memberships;", conn)
payments = pd.read_sql_query("SELECT * FROM payments;", conn)
complaints = pd.read_sql_query("SELECT * FROM complaints;", conn)

## Inventaire des anomalies à traiter

Avant d’appliquer le moindre nettoyage, on dresse une liste précise des problèmes observés dans la base :

- formats de dates hétérogènes,
- statuts numériques non documentés,
- statuts textuels incohérents,
- valeurs manquantes ou vides,
- doublons dans `payments`,
- lignes orphelines sur certaines clés de jointure,
- variations de casse dans `currency`, `country`, `status`, `type`,
- absence de clé primaire déclarée sur `memberships`.

L’objectif ici est de documenter ce qui doit être corrigé, conservé, ou simplement flaggé.

In [2]:
# Fonction utilitaire pour mesurer les valeurs manquantes
def missing_stats(df, table_name):
    stats = pd.DataFrame({
        "table": [table_name],
        "rows": [len(df)],
        "missing_cells": [df.isna().sum().sum()],
        "missing_ratio": [df.isna().sum().sum() / (df.shape[0] * df.shape[1]) if df.shape[0] and df.shape[1] else 0]
    })
    return stats

# Résumé des valeurs manquantes par table
missing_summary = pd.concat([
    missing_stats(users, "users"),
    missing_stats(subscriptions, "subscriptions"),
    missing_stats(memberships, "memberships"),
    missing_stats(payments, "payments"),
    missing_stats(complaints, "complaints"),
], ignore_index=True)

missing_summary

,table,rows,missing_cells,missing_ratio
0,users,2001,1862,0.116317
1,subscriptions,400,0,0.000000
2,memberships,1083,1343,0.177153
3,payments,7277,8814,0.121121
4,complaints,1213,1465,0.134194


### Lecture des valeurs manquantes

Le résumé des valeurs manquantes montre que la base contient un niveau de complétude variable selon les tables.

#### `users`
- **1862 cellules manquantes** sur **2001 lignes**
- ratio de valeurs manquantes : **11,63 %**

La table `users` contient donc plusieurs colonnes partiellement renseignées, ce qui devra être pris en compte sans supprimer systématiquement les lignes.

#### `subscriptions`
- **0 cellule manquante**
- ratio : **0 %**

Cette table est la plus propre sur le plan de la complétude, ce qui en fait une bonne base de jointure et d’agrégation.

#### `memberships`
- **1343 cellules manquantes** sur **1083 lignes**
- ratio de valeurs manquantes : **17,72 %**

Cette table est la plus fragile en complétude. Cela est cohérent avec la présence possible de memberships en cours, de dates de sortie absentes, ou de champs métier volontairement non renseignés.

#### `payments`
- **8814 cellules manquantes** sur **7277 lignes**
- ratio : **12,11 %**

Le niveau de valeurs manquantes est élevé mais attendu sur certains champs, notamment lorsque le paiement a échoué ou n’a pas été capturé. Les champs à interpréter avec prudence seront surtout `captured_at` et `stripe_error_code`.

#### `complaints`
- **1465 cellules manquantes** sur **1213 lignes**
- ratio : **13,42 %**

Cette table contient probablement des champs naturellement vides selon l’état de traitement des réclamations, en particulier `resolved_at` et `resolution`.

#### Conséquence pour le nettoyage
Les valeurs manquantes ne doivent pas être traitées de manière uniforme :
- certaines sont structurelles et métier-dépendantes,
- d’autres devront être conservées comme indicateurs d’absence d’information,
- d’autres enfin pourront être imputées ou remplacées par un flag selon leur impact sur le scoring.

La suite du nettoyage devra donc distinguer :
- les champs critiques à conserver intacts,
- les champs imputables,
- et les champs dont l’absence est elle-même informative.

## Règles de nettoyage retenues

Les anomalies observées montrent que la base est incomplète, hétérogène et partiellement dégradée, mais exploitable si les traitements sont faits avec prudence.

Constats importants à garder en tête :
- `users` contient des formats de dates hétérogènes et un niveau de valeurs manquantes non négligeable ;
- `subscriptions` est la table la plus propre en termes de complétude ;
- `memberships` est la table la plus fragile sur le plan des valeurs manquantes ;
- `payments` contient des doublons stricts et des statuts textuels incohérents ;
- `complaints` contient des champs naturellement vides selon l’état d’avancement des réclamations.

Principe général :
- ne jamais modifier la base brute ;
- conserver les colonnes brutes quand la normalisation peut faire perdre de l’information ;
- ajouter des flags d’anomalie plutôt que supprimer brutalement ;
- normaliser les formats textuels et temporels ;
- ne pas inventer de valeur manquante quand l’absence porte elle-même une information métier.

In [3]:
# Normalisation générique du texte : suppression des espaces inutiles
def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    s = str(value).strip()
    s = re.sub(r"\s+", " ", s)
    return s

# Passage en minuscules pour comparaison logique
def normalize_text_lower(value):
    s = normalize_text(value)
    if pd.isna(s):
        return pd.NA
    return s.lower()

# Suppression des accents
def strip_accents(value):
    if pd.isna(value):
        return pd.NA
    s = normalize_text(value)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s

# Normalisation des devises : casse homogène
def normalize_currency(value):
    if pd.isna(value):
        return pd.NA
    s = normalize_text(value)
    return s.upper()

# Normalisation des pays : harmonisation des variantes observées
country_aliases = {
    "FR": "FR",
    "FRA": "FR",
    "FRANCE": "FR",
    "FRANCAIS": "FR",
    "FRANCAISE": "FR",
}

def normalize_country(value):
    if pd.isna(value):
        return pd.NA
    s = normalize_text(value)
    s = strip_accents(s).upper()
    return country_aliases.get(s, s)

# Conversion robuste des dates hétérogènes vers un format UTC cohérent
def parse_mixed_datetime(value):
    if pd.isna(value):
        return pd.NaT

    # Objet datetime déjà reconnu
    if isinstance(value, pd.Timestamp):
        if value.tzinfo is None:
            return value.tz_localize("UTC")
        return value.tz_convert("UTC")

    # Timestamps numériques
    if isinstance(value, (int, float)) and not pd.isna(value):
        try:
            return pd.to_datetime(value, unit="s", utc=True, errors="coerce")
        except Exception:
            return pd.NaT

    s = str(value).strip()

    # Timestamp numérique stocké en chaîne
    if re.fullmatch(r"\d{10}", s):
        return pd.to_datetime(int(s), unit="s", utc=True, errors="coerce")

    if re.fullmatch(r"\d{13}", s):
        return pd.to_datetime(int(s), unit="ms", utc=True, errors="coerce")

    # Dates françaises avec slash : dayfirst=True seulement ici
    if "/" in s:
        return pd.to_datetime(s, utc=True, errors="coerce", dayfirst=True)

    # ISO / général : ne pas forcer dayfirst
    return pd.to_datetime(s, utc=True, errors="coerce")

# Normalisation des statuts textuels avec fusion des variantes les plus évidentes
def normalize_status_text(value):
    if pd.isna(value):
        return pd.NA

    s = normalize_text_lower(value)
    s = strip_accents(s)

    replacements = {
        "suceeded": "succeeded",
        "success": "succeeded",
        "succeeded": "succeeded",
        "failed": "failed",
        "pending": "pending",
        "refunded": "refunded",
        "disputed": "disputed",
        "canceled": "canceled",
        "cancelled": "canceled",
        "open": "open",
        "in_progress": "in_progress",
        "escalated": "escalated",
        "resolved": "resolved",
        "closed": "closed",
        "access denied": "access_denied",
        "acces refuse": "access_denied",
    }

    return replacements.get(s, s)

# Normalisation propre des types de réclamations, avec leur propre vocabulaire
def normalize_complaint_type(value):
    if pd.isna(value):
        return pd.NA

    s = normalize_text_lower(value)
    s = strip_accents(s)

    replacements = {
        "access denied": "access_denied",
        "acces refuse": "access_denied",
        "billing issue": "billing_issue",
        "billing_issue": "billing_issue",
        "fraud suspicion": "fraud_suspicion",
        "fraud_suspicion": "fraud_suspicion",
        "owner unresponsive": "owner_unresponsive",
        "owner_unresponsive": "owner_unresponsive",
        "subscription inactive": "subscription_inactive",
        "subscription_inactive": "subscription_inactive",
        "wrong credentials": "wrong_credentials",
        "wrong_credentials": "wrong_credentials",
        "other": "other",
    }

    return replacements.get(s, s)

# Normalisation neutre des statuts numériques :
# on ne devine pas la signification métier, on fabrique seulement un libellé stable et explicite
def normalize_numeric_status(value, prefix="unknown"):
    if pd.isna(value):
        return f"{prefix}_nan"
    try:
        v = int(value)
        return f"{prefix}_{v}"
    except Exception:
        return f"{prefix}_invalid"

# Flag simple pour signaler les valeurs manquantes
def missing_flag(series):
    return series.isna().astype(int)

## Reverse-engineering des statuts numériques

Certaines tables utilisent des statuts numériques non documentés, en particulier `users`, `subscriptions` et `memberships`.

Il ne faut pas inventer leur signification.  
On doit plutôt :
- observer leurs distributions,
- les relier aux dates, aux motifs et aux événements associés,
- produire un codebook interne prudent,
- ne valider une étiquette métier que si elle est soutenue par les données.

Si une signification reste incertaine, on conserve un libellé neutre du type `unknown_5`.

In [4]:
# Fonction d'analyse des statuts numériques pour préparer leur interprétation métier
def numeric_status_profile(df, table_name, status_col="status"):
    profile = (
        df.groupby(status_col, dropna=False)
          .size()
          .reset_index(name="count")
          .rename(columns={status_col: "status_code"})
    )
    profile["table"] = table_name
    return profile[["table", "status_code", "count"]]

# Profils des statuts numériques pour les tables concernées
users_status_profile = numeric_status_profile(users, "users")
subscriptions_status_profile = numeric_status_profile(subscriptions, "subscriptions")
memberships_status_profile = numeric_status_profile(memberships, "memberships")

# Fusion des profils pour lecture globale
status_profiles = pd.concat(
    [users_status_profile, subscriptions_status_profile, memberships_status_profile],
    ignore_index=True
)

status_profiles

,table,status_code,count
0,users,-1.0,38
1,users,0.0,1325
2,users,1.0,179
3,users,2.0,107
4,users,3.0,110
5,users,4.0,198
6,users,99.0,19
7,users,NaN,25
8,subscriptions,0.0,229
9,subscriptions,1.0,46


## Application du nettoyage

Le nettoyage est appliqué table par table pour conserver les colonnes brutes et produire des colonnes normalisées parallèles.

Les résultats de l’analyse des statuts numériques imposent une approche prudente :
- `users.status` contient des codes `0` à `4`, plus des valeurs atypiques `-1`, `99` et `NaN` ;
- `subscriptions.status` contient des codes `0` à `3` ;
- `memberships.status` contient des codes `0` à `7`.

À ce stade, il ne faut pas inventer la signification métier exacte de ces codes.  
On conserve donc les valeurs brutes, on normalise les codes de manière neutre, et on ajoute des flags pour les cas atypiques.

Règles retenues :
- les dates sont converties en UTC dans des colonnes dédiées ;
- les libellés textuels sont harmonisés ;
- les colonnes sensibles conservent leur version brute ;
- les doublons stricts de `payments` sont flaggés puis retirés de la couche propre ;
- les lignes orphelines sont conservées mais signalées ;
- les statuts numériques sont gardés sous forme brute et remplacés par des libellés neutres tant que leur sens métier reste incertain.

## Nettoyage de `users`

La table `users` contient plusieurs particularités à traiter avec prudence :
- `signup_date` mélange plusieurs formats de dates, dont certains avec fuseau horaire et d’autres sous forme de timestamp numérique ;
- `last_seen` semble plus homogène, mais doit tout de même être converti ;
- `country` contient des variantes de casse ou d’écriture ;
- `referral_code` est parfois vide, ce qui peut être une information utile en soi ;
- `status` est numérique et comprend des valeurs atypiques (`-1`, `99`, `NaN`) qu’il ne faut pas interpréter trop vite.

On conserve donc les colonnes brutes, on crée des colonnes normalisées parallèles, et on ajoute des flags pour signaler les cas inhabituels sans perdre l’information d’origine.

In [5]:
# Copie de travail pour ne jamais modifier la table brute
users_clean = users.copy()

# Conservation des valeurs brutes avant normalisation
users_clean["signup_date_raw"] = users_clean["signup_date"]
users_clean["last_seen_raw"] = users_clean["last_seen"]
users_clean["status_raw"] = users_clean["status"]

# Normalisation des dates
users_clean["signup_at_utc"] = users_clean["signup_date"].apply(parse_mixed_datetime)
users_clean["last_seen_at_utc"] = users_clean["last_seen"].apply(parse_mixed_datetime)

# Normalisation textuelle
users_clean["country_raw"] = users_clean["country"]
users_clean["country"] = users_clean["country"].apply(normalize_country)

# Préservation de l'information sur l'absence de code de parrainage
users_clean["referral_code_missing"] = missing_flag(users_clean["referral_code"])

# Harmonisation simple du champ phone_prefix
users_clean["phone_prefix"] = users_clean["phone_prefix"].apply(normalize_text)

# Statut numérique conservé, mais normalisé sous forme neutre
users_clean["status_norm"] = users_clean["status"].apply(lambda x: normalize_numeric_status(x, prefix="user_status"))

# Flag des statuts atypiques déjà observés
users_clean["status_is_anomalous"] = users_clean["status"].isin([-1, 99]) | users_clean["status"].isna()

## Nettoyage de `subscriptions`

Cette table est la plus propre en termes de complétude, mais elle doit tout de même être normalisée :
- dates converties en UTC,
- devise harmonisée,
- statut conservé brut avant interprétation,
- `owner_id` testé et flaggé en cas d’absence de correspondance.

Les statuts observés sont limités à `0` à `3`, ce qui permet une normalisation neutre sans supposer leur signification métier.

In [6]:
subscriptions_clean = subscriptions.copy()

# Conservation des valeurs brutes avant normalisation
subscriptions_clean["created_at_raw"] = subscriptions_clean["created_at"]
subscriptions_clean["status_raw"] = subscriptions_clean["status"]
subscriptions_clean["currency_raw"] = subscriptions_clean["currency"]

# Normalisation de la date de création
subscriptions_clean["created_at_utc"] = subscriptions_clean["created_at"].apply(parse_mixed_datetime)

# Harmonisation de la devise
subscriptions_clean["currency"] = subscriptions_clean["currency"].apply(normalize_currency)

# Normalisation neutre du statut numérique
subscriptions_clean["status_norm"] = subscriptions_clean["status"].apply(lambda x: normalize_numeric_status(x, prefix="subscription_status"))

# Flag des owner_id orphelins
valid_user_ids = set(users["id"].dropna().astype(int))
subscriptions_clean["owner_is_orphan"] = ~subscriptions_clean["owner_id"].isin(valid_user_ids)

## Nettoyage de `memberships`

Cette table est structurellement importante mais fragile :
- pas de clé primaire explicite dans le schéma affiché,
- beaucoup de valeurs manquantes,
- statut numérique non documenté,
- `left_at` peut être vide de façon légitime.

Les codes de statut observés vont de `0` à `7`.  
On les conserve donc comme codes bruts, avec une normalisation neutre, sans leur attribuer de signification métier prématurée.

In [7]:
memberships_clean = memberships.copy()

# Conservation des colonnes brutes
memberships_clean["joined_at_raw"] = memberships_clean["joined_at"]
memberships_clean["left_at_raw"] = memberships_clean["left_at"]
memberships_clean["status_raw"] = memberships_clean["status"]
memberships_clean["reason_raw"] = memberships_clean["reason"]

# Normalisation temporelle
memberships_clean["joined_at_utc"] = memberships_clean["joined_at"].apply(parse_mixed_datetime)
memberships_clean["left_at_utc"] = memberships_clean["left_at"].apply(parse_mixed_datetime)

# Harmonisation textuelle du motif si besoin
memberships_clean["reason"] = memberships_clean["reason"].apply(normalize_text_lower)

# Normalisation neutre du statut numérique
memberships_clean["status_norm"] = memberships_clean["status"].apply(lambda x: normalize_numeric_status(x, prefix="membership_status"))

# Flags de qualité
memberships_clean["left_at_missing"] = memberships_clean["left_at"].isna().astype(int)
memberships_clean["user_is_orphan"] = ~memberships_clean["user_id"].isin(valid_user_ids)
valid_subscription_ids = set(subscriptions["id"].dropna().astype(int))
memberships_clean["subscription_is_orphan"] = ~memberships_clean["subscription_id"].isin(valid_subscription_ids)

## Nettoyage de `payments`

La table `payments` est la plus sensible du projet :
- plusieurs doublons stricts,
- statuts textuels incohérents,
- variations de casse,
- fautes de frappe,
- dates avec et sans fuseau horaire,
- lignes orphelines sur les clés de jointure.

Cette table doit donc être nettoyée avec prudence, tout en conservant un maximum de trace brute.

In [8]:
payments_clean = payments.copy()

# Conservation des champs bruts
payments_clean["status_raw"] = payments_clean["status"]
payments_clean["created_at_raw"] = payments_clean["created_at"]
payments_clean["captured_at_raw"] = payments_clean["captured_at"]
payments_clean["currency_raw"] = payments_clean["currency"]

# Normalisation des statuts et devises
payments_clean["status"] = payments_clean["status"].apply(normalize_status_text)
payments_clean["currency"] = payments_clean["currency"].apply(normalize_currency)

# Conversion temporelle
payments_clean["created_at_utc"] = payments_clean["created_at"].apply(parse_mixed_datetime)
payments_clean["captured_at_utc"] = payments_clean["captured_at"].apply(parse_mixed_datetime)

# Flags d'anomalie
payments_clean["user_is_orphan"] = ~payments_clean["user_id"].isin(valid_user_ids)
payments_clean["subscription_is_orphan"] = ~payments_clean["subscription_id"].isin(valid_subscription_ids)

# Duplication contolée de payments                                                                                      
# Détection de doublons stricts à partir des colonnes métier observées
dup_cols = [
    "user_id", "subscription_id", "amount_cents", "fee_cents",
    "status_raw", "created_at", "captured_at", "currency_raw", "stripe_error_code"
]

payments_clean["is_strict_duplicate"] = payments_clean.duplicated(subset=dup_cols, keep="first")

# On conserve uniquement la première occurrence des doublons stricts
payments_clean_dedup = payments_clean.loc[~payments_clean["is_strict_duplicate"]].copy()                                                                                           

## Nettoyage de `complaints`

Les réclamations sont métier-dépendantes :
- un statut `open` ou `in_progress` n’est pas une erreur,
- un `resolved_at` vide peut être normal,
- les variantes de casse et de langue doivent être harmonisées.

Cette table doit rester exploitable pour le scoring sans perdre la notion d’état de traitement.

In [9]:
complaints_clean = complaints.copy()

# Conservation des valeurs brutes
complaints_clean["type_raw"] = complaints_clean["type"]
complaints_clean["status_raw"] = complaints_clean["status"]
complaints_clean["created_at_raw"] = complaints_clean["created_at"]
complaints_clean["resolved_at_raw"] = complaints_clean["resolved_at"]
complaints_clean["resolution_raw"] = complaints_clean["resolution"]

# Normalisation du type de réclamation avec son vocabulaire propre
complaints_clean["type"] = complaints_clean["type"].apply(normalize_complaint_type)

# Normalisation séparée du statut et de la résolution
complaints_clean["status"] = complaints_clean["status"].apply(normalize_status_text)
complaints_clean["resolution"] = complaints_clean["resolution"].apply(normalize_text_lower)

# Conversion temporelle
complaints_clean["created_at_utc"] = complaints_clean["created_at"].apply(parse_mixed_datetime)
complaints_clean["resolved_at_utc"] = complaints_clean["resolved_at"].apply(parse_mixed_datetime)

# Flags d'anomalie
complaints_clean["reporter_is_orphan"] = ~complaints_clean["reporter_id"].isin(valid_user_ids)
complaints_clean["target_is_orphan"] = ~complaints_clean["target_id"].isin(valid_user_ids)
complaints_clean["subscription_is_orphan"] = ~complaints_clean["subscription_id"].isin(valid_subscription_ids)

## Validation des tables nettoyées avant export

Avant d’écrire le SQLite propre, on vérifie que chaque table nettoyée contient bien :
- les bonnes colonnes,
- les bonnes tailles,
- les bonnes valeurs de tête,
- des dates interprétées correctement,
- aucun mélange entre tables.

Cette étape est essentielle pour éviter d’exporter une base structurellement correcte mais sémantiquement corrompue.

In [10]:
# Vérification des tailles attendues
expected_counts = {
    "users": 2001,
    "subscriptions": 400,
    "memberships": 1083,
    "payments": 7251,
    "complaints": 1213,
}

assert len(users_clean) == expected_counts["users"], "Taille inattendue pour users_clean"
assert len(subscriptions_clean) == expected_counts["subscriptions"], "Taille inattendue pour subscriptions_clean"
assert len(memberships_clean) == expected_counts["memberships"], "Taille inattendue pour memberships_clean"
assert len(payments_clean_dedup) == expected_counts["payments"], "Taille inattendue pour payments_clean_dedup"
assert len(complaints_clean) == expected_counts["complaints"], "Taille inattendue pour complaints_clean"

# Vérification visuelle obligatoire avant export
display(users_clean.head(3))
display(subscriptions_clean.head(3))
display(memberships_clean.head(3))
display(payments_clean_dedup.head(3))
display(complaints_clean.head(3))

# Vérification de quelques conversions sensibles
sample_users = users_clean.loc[users_clean["id"].isin([1, 5]), ["id", "signup_date_raw", "signup_at_utc"]]
display(sample_users)

# La date ISO doit rester correcte : 2021-02-09 21:26:47 -> 2021-02-09T21:26:47Z
assert users_clean.loc[users_clean["id"] == 1, "signup_at_utc"].iloc[0] == pd.Timestamp("2021-02-09 21:26:47+00:00")

# La date française doit être correctement lue : 21/07/2020 08:54 -> 2020-07-21T08:54:00Z
assert users_clean.loc[users_clean["id"] == 5, "signup_at_utc"].iloc[0] == pd.Timestamp("2020-07-21 08:54:00+00:00")

# Vérification de la séparation type / status dans complaints
display(complaints_clean[["type_raw", "type", "status_raw", "status"]].head(5))

# Contrôles anti-mélange entre tables
assert not subscriptions_clean["brand"].astype(str).str.contains("@", na=False).any()
assert not complaints_clean["type"].astype(str).str.contains("@", na=False).any()
assert pd.to_numeric(subscriptions_clean["owner_id"], errors="coerce").notna().all()
assert pd.to_numeric(memberships_clean["user_id"], errors="coerce").notna().all()
assert pd.to_numeric(payments_clean_dedup["amount_cents"], errors="coerce").notna().all()
assert pd.to_numeric(complaints_clean["reporter_id"], errors="coerce").notna().all()

,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix,signup_date_raw,last_seen_raw,status_raw,signup_at_utc,last_seen_at_utc,country_raw,referral_code_missing,status_norm,status_is_anomalous
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1.0,2021-04-28 14:23:00,NaN,+49,2021-02-09 21:26:47,2021-04-28 14:23:00,1.0,2021-02-09 21:26:47+00:00,2021-04-28 14:23:00+00:00,IT,1,user_status_1,False
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0.0,2023-01-05 17:03:20,NaN,+41,2022-01-09T06:49:44Z,2023-01-05 17:03:20,0.0,2022-01-09 06:49:44+00:00,2023-01-05 17:03:20+00:00,CH,1,user_status_0,False
2,3,user_3@outlook.com,BE,1584765297,1.0,2021-07-30 05:38:37,27cb14dc,+32,1584765297,2021-07-30 05:38:37,1.0,2020-03-21 04:34:57+00:00,2021-07-30 05:38:37+00:00,BE,0,user_status_1,False


,id,brand,owner_id,created_at,status,max_slots,price_cents,currency,created_at_raw,status_raw,currency_raw,created_at_utc,status_norm,owner_is_orphan
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,EUR,2023-01-14 23:23:57,0,eur,2023-01-14 23:23:57+00:00,subscription_status_0,False
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD,2023-07-16 21:39:51,2,USD,2023-07-16 21:39:51+00:00,subscription_status_2,False
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR,2024-10-02 13:54:25,0,EUR,2024-10-02 13:54:25+00:00,subscription_status_0,False


,id,user_id,subscription_id,status,joined_at,left_at,reason,joined_at_raw,left_at_raw,status_raw,reason_raw,joined_at_utc,left_at_utc,status_norm,left_at_missing,user_is_orphan,subscription_is_orphan
0,563,1485,231,1,2024-12-06 21:23:01,NaN,NaN,2024-12-06 21:23:01,NaN,1,NaN,2024-12-06 21:23:01+00:00,NaT,membership_status_1,1,False,False
1,1042,690,399,2,2024-09-23 02:25:46,2025-03-31 22:56:44,fraud,2024-09-23 02:25:46,2025-03-31 22:56:44,2,fraud,2024-09-23 02:25:46+00:00,2025-03-31 22:56:44+00:00,membership_status_2,0,False,False
2,72,56,31,1,2025-02-15 18:19:15,NaN,NaN,2025-02-15 18:19:15,NaN,1,NaN,2025-02-15 18:19:15+00:00,NaT,membership_status_1,1,False,False


,id,user_id,subscription_id,amount_cents,fee_cents,status,created_at,captured_at,currency,stripe_error_code,status_raw,created_at_raw,captured_at_raw,currency_raw,created_at_utc,captured_at_utc,user_is_orphan,subscription_is_orphan,is_strict_duplicate
0,1,1485,231,1322,78,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,NaN,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,2024-12-04 19:23:01+00:00,2024-12-07 14:23:01+00:00,False,False,False
1,2,1485,231,366,37,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,EUR,NaN,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,eur,2025-01-03 20:23:01+00:00,2025-01-05 11:23:01+00:00,False,False,False
2,3,1485,231,238,28,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,NaN,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,2025-02-07 21:23:01+00:00,2025-02-10 04:23:01+00:00,False,False,False


,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution,type_raw,status_raw,created_at_raw,resolved_at_raw,resolution_raw,created_at_utc,resolved_at_utc,reporter_is_orphan,target_is_orphan,subscription_is_orphan
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN,billing_issue,open,2023-09-06 08:02:33,NaN,NaN,2023-09-06 08:02:33+00:00,NaT,False,False,False
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN,2024-07-30 12:44:00+00:00,NaT,False,False,False
2,3,1361,672,124,access_denied,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,,2022-10-01 21:19:13+00:00,2024-01-27 20:51:20+00:00,False,False,False


,id,signup_date_raw,signup_at_utc
0,1,2021-02-09 21:26:47,2021-02-09 21:26:47+00:00
4,5,21/07/2020 08:54,2020-07-21 08:54:00+00:00


,type_raw,type,status_raw,status
0,billing_issue,billing_issue,open,open
1,subscription_inactive,subscription_inactive,escalated,escalated
2,Accès refusé,access_denied,resolved,resolved
3,ACCESS_DENIED,access_denied,RESOLVED,resolved
4,ACCESS_DENIED,access_denied,closed,closed


### Validation finale des tables nettoyées avant export

La validation des tables nettoyées confirme que la structure est cohérente et prête pour l’export.

Les contrôles réalisés montrent que :
- les tables restent bien séparées et ne se mélangent pas ;
- les colonnes brutes sont conservées dans les champs `_raw` ;
- les colonnes normalisées sont présentes et lisibles ;
- les dates ont été correctement converties en UTC ;
- les statuts numériques sont conservés sous une forme neutre ;
- les réclamations distinguent correctement `type`, `status` et `resolution` ;
- les flags d’anomalie et d’orphelins sont bien calculés.

Les exemples contrôlés montrent que :
- `2021-02-09 21:26:47` devient `2021-02-09 21:26:47+00:00` ;
- `21/07/2020 08:54` devient `2020-07-21 08:54:00+00:00` ;
- `Accès refusé` et `ACCESS_DENIED` sont bien normalisés en `access_denied` ;
- les dates ISO et les dates françaises sont traitées différemment de façon correcte.

La base nettoyée peut maintenant être exportée dans un SQLite propre pour servir de base de travail à l’étape suivante.

## Export de la base nettoyée avec schéma explicite

La version nettoyée doit être persistée dans un second fichier SQLite afin de conserver la structure relationnelle de la base et de faciliter la suite du projet.

Ce choix est volontaire :
- il préserve les relations entre tables ;
- il permet de refaire des jointures SQL simplement ;
- il évite la perte de structure qu’imposerait un export uniquement en CSV ;
- il reste cohérent avec le format source du projet.

Nous ne déclarons pas de contraintes de clés étrangères dans cette base nettoyée, car le dataset contient des lignes orphelines observées lors de l’exploration. Une contrainte stricte ferait échouer l’insertion ou forcerait la suppression de données utiles pour le scoring.

La base brute reste intacte dans `data/raw/`; seule une copie nettoyée est écrite dans `data/processed/`.

In [11]:
# Chemin de sortie pour la base nettoyée
clean_db_path = Path("H:/Ecole/ECE/CESURE/ALTERNANCE/Sharesub.com/TEST_TECHNIQUE/risk-monitor/data/processed/risk_monitor_clean.sqlite")

# S'assurer que le dossier de sortie existe
clean_db_path.parent.mkdir(parents=True, exist_ok=True)

# Recréer le fichier SQLite propre à partir de zéro
if clean_db_path.exists():
    clean_db_path.unlink()

# Création de la connexion SQLite de sortie
clean_conn = sqlite3.connect(clean_db_path)

# Schéma explicite des tables.
# memberships n'avait pas de PRIMARY KEY explicite dans le schéma source, donc on ne l'invente pas ici.
schema_sql = """
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    email TEXT,
    country TEXT,
    signup_date TEXT,
    status INTEGER,
    last_seen TEXT,
    referral_code TEXT,
    phone_prefix TEXT,
    signup_date_raw TEXT,
    last_seen_raw TEXT,
    status_raw INTEGER,
    signup_at_utc TEXT,
    last_seen_at_utc TEXT,
    country_raw TEXT,
    referral_code_missing INTEGER,
    status_norm TEXT,
    status_is_anomalous INTEGER
);

CREATE TABLE subscriptions (
    id INTEGER PRIMARY KEY,
    brand TEXT,
    owner_id INTEGER,
    created_at TEXT,
    status INTEGER,
    max_slots INTEGER,
    price_cents INTEGER,
    currency TEXT,
    created_at_raw TEXT,
    status_raw INTEGER,
    currency_raw TEXT,
    created_at_utc TEXT,
    status_norm TEXT,
    owner_is_orphan INTEGER
);

CREATE TABLE memberships (
    id INTEGER,
    user_id INTEGER,
    subscription_id INTEGER,
    status INTEGER,
    joined_at TEXT,
    left_at TEXT,
    reason TEXT,
    joined_at_raw TEXT,
    left_at_raw TEXT,
    status_raw INTEGER,
    reason_raw TEXT,
    joined_at_utc TEXT,
    left_at_utc TEXT,
    status_norm TEXT,
    left_at_missing INTEGER,
    user_is_orphan INTEGER,
    subscription_is_orphan INTEGER
);

CREATE TABLE payments (
    id INTEGER PRIMARY KEY,
    user_id INTEGER,
    subscription_id INTEGER,
    amount_cents INTEGER,
    fee_cents INTEGER,
    status TEXT,
    created_at TEXT,
    captured_at TEXT,
    currency TEXT,
    stripe_error_code TEXT,
    status_raw TEXT,
    created_at_raw TEXT,
    captured_at_raw TEXT,
    currency_raw TEXT,
    created_at_utc TEXT,
    captured_at_utc TEXT,
    user_is_orphan INTEGER,
    subscription_is_orphan INTEGER
);

CREATE TABLE complaints (
    id INTEGER PRIMARY KEY,
    reporter_id INTEGER,
    target_id INTEGER,
    subscription_id INTEGER,
    type TEXT,
    status TEXT,
    created_at TEXT,
    resolved_at TEXT,
    resolution TEXT,
    type_raw TEXT,
    status_raw TEXT,
    created_at_raw TEXT,
    resolved_at_raw TEXT,
    resolution_raw TEXT,
    created_at_utc TEXT,
    resolved_at_utc TEXT,
    reporter_is_orphan INTEGER,
    target_is_orphan INTEGER,
    subscription_is_orphan INTEGER
);
"""

clean_conn.executescript(schema_sql)

# Préparation des colonnes d'export
users_export_cols = [
    "id", "email", "country", "signup_date", "status", "last_seen", "referral_code", "phone_prefix",
    "signup_date_raw", "last_seen_raw", "status_raw", "signup_at_utc", "last_seen_at_utc",
    "country_raw", "referral_code_missing", "status_norm", "status_is_anomalous"
]

subscriptions_export_cols = [
    "id", "brand", "owner_id", "created_at", "status", "max_slots", "price_cents", "currency",
    "created_at_raw", "status_raw", "currency_raw", "created_at_utc", "status_norm", "owner_is_orphan"
]

memberships_export_cols = [
    "id", "user_id", "subscription_id", "status", "joined_at", "left_at", "reason",
    "joined_at_raw", "left_at_raw", "status_raw", "reason_raw",
    "joined_at_utc", "left_at_utc", "status_norm",
    "left_at_missing", "user_is_orphan", "subscription_is_orphan"
]

payments_export_cols = [
    "id", "user_id", "subscription_id", "amount_cents", "fee_cents", "status",
    "created_at", "captured_at", "currency", "stripe_error_code",
    "status_raw", "created_at_raw", "captured_at_raw", "currency_raw",
    "created_at_utc", "captured_at_utc", "user_is_orphan", "subscription_is_orphan"
]

complaints_export_cols = [
    "id", "reporter_id", "target_id", "subscription_id", "type", "status",
    "created_at", "resolved_at", "resolution",
    "type_raw", "status_raw", "created_at_raw", "resolved_at_raw", "resolution_raw",
    "created_at_utc", "resolved_at_utc",
    "reporter_is_orphan", "target_is_orphan", "subscription_is_orphan"
]

# Préparation des DataFrames pour l'export :
# - ordre stable des colonnes
# - dates UTC transformées en texte ISO-8601 pour rester lisibles dans SQLite
def prepare_export_df(df, cols, datetime_cols=None):
    out = df.loc[:, cols].copy()
    datetime_cols = datetime_cols or []
    for col in datetime_cols:
        out[col] = pd.to_datetime(out[col], utc=True, errors="coerce").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return out

users_export = prepare_export_df(users_clean, users_export_cols, datetime_cols=["signup_at_utc", "last_seen_at_utc"])
subscriptions_export = prepare_export_df(subscriptions_clean, subscriptions_export_cols, datetime_cols=["created_at_utc"])
memberships_export = prepare_export_df(memberships_clean, memberships_export_cols, datetime_cols=["joined_at_utc", "left_at_utc"])
payments_export = prepare_export_df(payments_clean_dedup, payments_export_cols, datetime_cols=["created_at_utc", "captured_at_utc"])
complaints_export = prepare_export_df(complaints_clean, complaints_export_cols, datetime_cols=["created_at_utc", "resolved_at_utc"])

# Vérification finale des colonnes avant écriture
assert users_export.columns.tolist() == users_export_cols
assert subscriptions_export.columns.tolist() == subscriptions_export_cols
assert memberships_export.columns.tolist() == memberships_export_cols
assert payments_export.columns.tolist() == payments_export_cols
assert complaints_export.columns.tolist() == complaints_export_cols

# Vérification visuelle obligatoire avant écriture
display(users_export.head(3))
display(subscriptions_export.head(3))
display(memberships_export.head(3))
display(payments_export.head(3))
display(complaints_export.head(3))

# Écriture des tables nettoyées
users_export.to_sql("users", clean_conn, if_exists="append", index=False)
subscriptions_export.to_sql("subscriptions", clean_conn, if_exists="append", index=False)
memberships_export.to_sql("memberships", clean_conn, if_exists="append", index=False)
payments_export.to_sql("payments", clean_conn, if_exists="append", index=False)
complaints_export.to_sql("complaints", clean_conn, if_exists="append", index=False)

clean_conn.commit()
clean_conn.close()

,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix,signup_date_raw,last_seen_raw,status_raw,signup_at_utc,last_seen_at_utc,country_raw,referral_code_missing,status_norm,status_is_anomalous
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1.0,2021-04-28 14:23:00,NaN,+49,2021-02-09 21:26:47,2021-04-28 14:23:00,1.0,2021-02-09T21:26:47Z,2021-04-28T14:23:00Z,IT,1,user_status_1,False
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0.0,2023-01-05 17:03:20,NaN,+41,2022-01-09T06:49:44Z,2023-01-05 17:03:20,0.0,2022-01-09T06:49:44Z,2023-01-05T17:03:20Z,CH,1,user_status_0,False
2,3,user_3@outlook.com,BE,1584765297,1.0,2021-07-30 05:38:37,27cb14dc,+32,1584765297,2021-07-30 05:38:37,1.0,2020-03-21T04:34:57Z,2021-07-30T05:38:37Z,BE,0,user_status_1,False


,id,brand,owner_id,created_at,status,max_slots,price_cents,currency,created_at_raw,status_raw,currency_raw,created_at_utc,status_norm,owner_is_orphan
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,EUR,2023-01-14 23:23:57,0,eur,2023-01-14T23:23:57Z,subscription_status_0,False
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD,2023-07-16 21:39:51,2,USD,2023-07-16T21:39:51Z,subscription_status_2,False
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR,2024-10-02 13:54:25,0,EUR,2024-10-02T13:54:25Z,subscription_status_0,False


,id,user_id,subscription_id,status,joined_at,left_at,reason,joined_at_raw,left_at_raw,status_raw,reason_raw,joined_at_utc,left_at_utc,status_norm,left_at_missing,user_is_orphan,subscription_is_orphan
0,563,1485,231,1,2024-12-06 21:23:01,NaN,NaN,2024-12-06 21:23:01,NaN,1,NaN,2024-12-06T21:23:01Z,NaN,membership_status_1,1,False,False
1,1042,690,399,2,2024-09-23 02:25:46,2025-03-31 22:56:44,fraud,2024-09-23 02:25:46,2025-03-31 22:56:44,2,fraud,2024-09-23T02:25:46Z,2025-03-31T22:56:44Z,membership_status_2,0,False,False
2,72,56,31,1,2025-02-15 18:19:15,NaN,NaN,2025-02-15 18:19:15,NaN,1,NaN,2025-02-15T18:19:15Z,NaN,membership_status_1,1,False,False


,id,user_id,subscription_id,amount_cents,fee_cents,status,created_at,captured_at,currency,stripe_error_code,status_raw,created_at_raw,captured_at_raw,currency_raw,created_at_utc,captured_at_utc,user_is_orphan,subscription_is_orphan
0,1,1485,231,1322,78,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,NaN,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,2024-12-04T19:23:01Z,2024-12-07T14:23:01Z,False,False
1,2,1485,231,366,37,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,EUR,NaN,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,eur,2025-01-03T20:23:01Z,2025-01-05T11:23:01Z,False,False
2,3,1485,231,238,28,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,NaN,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,2025-02-07T21:23:01Z,2025-02-10T04:23:01Z,False,False


,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution,type_raw,status_raw,created_at_raw,resolved_at_raw,resolution_raw,created_at_utc,resolved_at_utc,reporter_is_orphan,target_is_orphan,subscription_is_orphan
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN,billing_issue,open,2023-09-06 08:02:33,NaN,NaN,2023-09-06T08:02:33Z,NaN,False,False,False
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN,2024-07-30T12:44:00Z,NaN,False,False,False
2,3,1361,672,124,access_denied,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,,2022-10-01T21:19:13Z,2024-01-27T20:51:20Z,False,False,False


### Vérification de l'export

La base nettoyée a été recréée à partir des fonctions corrigées.

Ce point est essentiel :
- les dates ISO ne doivent plus être inversées ;
- les dates françaises doivent rester correctement interprétées ;
- les types de réclamations doivent être normalisés avec leur vocabulaire propre ;
- la base SQLite propre devient la source de travail pour la suite du projet.

La base brute reste inchangée dans `data/raw/`.

In [12]:
# Vérification rapide du contenu de la base nettoyée
verify_conn = sqlite3.connect(clean_db_path)

# Vérification des tables
tables_check = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    verify_conn
)

display(tables_check)

# Vérification des volumes
for table in ["users", "subscriptions", "memberships", "payments", "complaints"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table};", verify_conn)
    print(f"{table}: {count.iloc[0]['n']}")

# Vérification d'exemples de dates et de normalisation
print("\nAperçu de users :")
display(pd.read_sql_query("SELECT signup_date, signup_at_utc FROM users LIMIT 5;", verify_conn))

print("\nAperçu de subscriptions :")
display(pd.read_sql_query("SELECT brand, owner_id, created_at, created_at_utc FROM subscriptions LIMIT 5;", verify_conn))

print("\nAperçu de memberships :")
display(pd.read_sql_query("SELECT user_id, subscription_id, joined_at, joined_at_utc FROM memberships LIMIT 5;", verify_conn))

print("\nAperçu de payments :")
display(pd.read_sql_query("SELECT user_id, subscription_id, amount_cents, status, created_at, created_at_utc FROM payments LIMIT 5;", verify_conn))

print("\nAperçu de complaints :")
display(pd.read_sql_query("SELECT type_raw, type, status_raw, status FROM complaints LIMIT 5;", verify_conn))

# Vérification de l'absence de doublons stricts restants dans payments
duplicate_check = pd.read_sql_query(
    """
    SELECT user_id, subscription_id, amount_cents, fee_cents, status_raw, created_at_raw, captured_at_raw, currency_raw, stripe_error_code, COUNT(*) AS n
    FROM payments
    GROUP BY user_id, subscription_id, amount_cents, fee_cents, status_raw, created_at_raw, captured_at_raw, currency_raw, stripe_error_code
    HAVING COUNT(*) > 1;
    """,
    verify_conn
)

display(duplicate_check)

# Vérification du schéma enregistré dans SQLite
schema_check = pd.read_sql_query(
    "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name;",
    verify_conn
)

display(schema_check)

verify_conn.close()

,name
0,complaints
1,memberships
2,payments
3,subscriptions
4,users


users: 2001
subscriptions: 400
memberships: 1083
payments: 7251
complaints: 1213

Aperçu de users :


,signup_date,signup_at_utc
0,2021-02-09 21:26:47,2021-02-09T21:26:47Z
1,2022-01-09T06:49:44Z,2022-01-09T06:49:44Z
2,1584765297,2020-03-21T04:34:57Z
3,2020-07-13T23:53:58Z,2020-07-13T23:53:58Z
4,21/07/2020 08:54,2020-07-21T08:54:00Z



Aperçu de subscriptions :


,brand,owner_id,created_at,created_at_utc
0,Microsoft 365,1,2023-01-14 23:23:57,2023-01-14T23:23:57Z
1,HBO Max,1670,2023-07-16 21:39:51,2023-07-16T21:39:51Z
2,ChatGPT Plus,558,2024-10-02 13:54:25,2024-10-02T13:54:25Z
3,Microsoft 365,567,2023-10-26 23:03:50,2023-10-26T23:03:50Z
4,NordVPN,950,2024-11-30 08:35:40,2024-11-30T08:35:40Z


...

,user_id,subscription_id,amount_cents,status,created_at,created_at_utc
0,1485,231,1322,succeeded,2024-12-04T21:23:01+02:00,2024-12-04T19:23:01Z
1,1485,231,366,succeeded,2025-01-03T21:23:01+01:00,2025-01-03T20:23:01Z
2,1485,231,238,succeeded,2025-02-07 21:23:01,2025-02-07T21:23:01Z
3,1485,231,212,succeeded,2025-03-04 21:23:01,2025-03-04T21:23:01Z
4,1485,231,1325,succeeded,2025-04-03 21:23:01,2025-04-03T21:23:01Z



Aperçu de complaints :


,type_raw,type,status_raw,status
0,billing_issue,billing_issue,open,open
1,subscription_inactive,subscription_inactive,escalated,escalated
2,Accès refusé,access_denied,resolved,resolved
3,ACCESS_DENIED,access_denied,RESOLVED,resolved
4,ACCESS_DENIED,access_denied,closed,closed


,user_id,subscription_id,amount_cents,fee_cents,status_raw,created_at_raw,captured_at_raw,currency_raw,stripe_error_code,n


,name,sql
0,complaints,CREATE TABLE complaints (\n id INTEGER PRIM...
1,memberships,"CREATE TABLE memberships (\n id INTEGER,\n ..."
2,payments,CREATE TABLE payments (\n id INTEGER PRIMAR...
3,subscriptions,CREATE TABLE subscriptions (\n id INTEGER P...
4,users,CREATE TABLE users (\n id INTEGER PRIMARY K...


### Validation finale de la base nettoyée exportée

La base nettoyée exportée est cohérente et prête pour la suite du projet.

Les vérifications réalisées confirment que :
- les cinq tables attendues sont bien présentes ;
- les volumes correspondent à la base nettoyée attendue ;
- les dates ont été correctement converties en UTC et exportées au format ISO-8601 ;
- les valeurs d’origine sont conservées dans les colonnes `_raw` ;
- les champs normalisés sont bien séparés des champs bruts ;
- les réclamations distinguent correctement `type`, `status` et `resolution` ;
- le schéma SQLite est bien reconstruit de façon explicite ;
- aucun doublon strict résiduel n’a été détecté sur `payments`.

Les exemples contrôlés montrent que :
- `2021-02-09 21:26:47` devient `2021-02-09T21:26:47Z` ;
- `2022-01-09T06:49:44Z` reste correctement interprété ;
- `1584765297` est converti en date UTC ;
- `21/07/2020 08:54` devient `2020-07-21T08:54:00Z` ;
- `Accès refusé` et `ACCESS_DENIED` sont bien normalisés en `access_denied`.

La base nettoyée peut maintenant servir de base de travail pour l’étape 3 : construction des features de risque et calcul du score.

## Synthèse du nettoyage

Le nettoyage conserve les données sources intactes tout en produisant une version normalisée exploitable pour la suite du projet.

Décisions principales :
- les dates sont converties en UTC dans des colonnes dédiées ;
- les libellés textuels sont harmonisés ;
- les colonnes sensibles conservent leur version brute ;
- les doublons stricts de `payments` sont supprimés après traçage ;
- les lignes orphelines sont conservées mais signalées ;
- les statuts numériques sont conservés bruts et normalisés sous des libellés neutres ;
- les valeurs ambiguës ne sont pas inventées : elles restent visibles sous forme brute ou via un flag ;
- les dates normalisées sont exportées en texte ISO-8601 UTC pour rester lisibles et portables dans SQLite ;
- les réclamations sont traitées avec un vocabulaire séparé pour `type` et `status`, afin de ne pas mélanger la catégorie du problème et son état de traitement.

Un second fichier SQLite nettoyé a été exporté dans `data/processed/risk_monitor_clean.sqlite`.
Ce choix est volontaire car il permet de conserver la structure relationnelle de la base, de faciliter les jointures pour le scoring et l’interface, et de garder une source propre, cohérente et réutilisable pour les étapes suivantes.

Nous n’avons pas déclaré de clés étrangères dans la base nettoyée, car des lignes orphelines existent dans le dataset source et doivent être conservées comme signal d’anomalie plutôt que supprimées.

La base brute reste intacte dans `data/raw/`.
La base nettoyée devient la référence de travail pour la suite du projet, notamment pour la construction des features, le scoring et l’interface.